In [1]:
import pandas as pd
from xgboost import XGBRegressor 

In [17]:
df = pd.read_csv("D:\\Crop_yield_system\\data\\Processed\\features_engineered_v2.csv")
df.head()

,State,Year,Area,Season,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,...,SOC_Variability,SOC_Stability,Season_Num,Year_Season,Region,Agro_Zone,Area_Rain,NDVI_div_Rain,Rain_Anomaly,Temp_Anomaly
0,Chandigarh,2011,600,Kharif,28.545820,32.636721,24.651230,1120.70,2415.25,73.597787,...,0.180658,5.535331,1,2011_Kharif,North,Urban,672420.0,0.000451,0.000000,0.000000
1,Chandigarh,2011,600,Rabi,18.842547,26.398585,13.192453,81.17,3551.99,46.769575,...,0.180658,5.535331,2,2011_Rabi,North,Urban,48702.0,0.004723,-519.765000,-4.851636
2,Chandigarh,2012,575,Kharif,30.871885,35.985164,26.062869,761.14,2549.54,58.650328,...,0.180658,5.535331,1,2012_Kharif,North,Urban,437655.5,0.000601,106.803333,4.785135
3,Chandigarh,2012,575,Rabi,18.925164,26.774977,13.223897,87.20,3528.42,41.122770,...,0.180658,5.535331,2,2012_Rabi,North,Urban,50140.0,0.004358,-425.352500,-5.371190
4,Chandigarh,2013,575,Kharif,28.820410,32.811475,24.909918,1174.10,2440.15,72.614426,...,0.180658,5.535331,1,2013_Kharif,North,Urban,675107.5,0.000445,529.238000,3.619245


In [7]:
TARGET = "Yield"     # change to "Production" if needed
y = df[TARGET]
X = df.drop(columns=[TARGET])


In [20]:

df.info()
df["Rain_CV"] = df["Rain_CV"].fillna(0)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002 entries, 0 to 2001
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2002 non-null   object 
 1   Year                    2002 non-null   int64  
 2   Area                    2002 non-null   int64  
 3   Season                  2002 non-null   object 
 4   T2M                     2002 non-null   float64
 5   T2M_MAX                 2002 non-null   float64
 6   T2M_MIN                 2002 non-null   float64
 7   PRECTOTCORR             2002 non-null   float64
 8   ALLSKY_SFC_SW_DWN       2002 non-null   float64
 9   RH2M                    2002 non-null   float64
 10  WS2M                    2002 non-null   float64
 11  NDVI_SeasonalMean       2002 non-null   float64
 12  Mean_SOC                2002 non-null   float64
 13  Median_SOC              2002 non-null   float64
 14  Min_SOC                 2002 non-null   

In [21]:
df.isna().sum()

State                     0
Year                      0
Area                      0
Season                    0
T2M                       0
T2M_MAX                   0
T2M_MIN                   0
PRECTOTCORR               0
ALLSKY_SFC_SW_DWN         0
RH2M                      0
WS2M                      0
NDVI_SeasonalMean         0
Mean_SOC                  0
Median_SOC                0
Min_SOC                   0
Max_SOC                   0
Std_SOC                   0
Temp_Range                0
Temp_Stress               0
GDD                       0
Rain_Per_Ha               0
Rain_CV                   0
Solar_Temp_Interaction    0
Wind_Temp_Interaction     0
NDVI_Level                0
NDVI_Rain_Ratio           0
NDVI_Temp_Interaction     0
SOC_Range                 0
SOC_Variability           0
SOC_Stability             0
Season_Num                0
Year_Season               0
Region                    0
Agro_Zone                 0
Area_Rain                 0
NDVI_div_Rain       

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Simple encoding
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols)
X_test_encoded  = pd.get_dummies(X_test,  columns=categorical_cols)

# Align columns (important!)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join="left", axis=1, fill_value=0)


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 2. FILE PATHS
# ============================================================
ENGINEERED_PATH = r"D:\Crop_yield_system\data\Processed\features_engineered_v2.csv"
RAW_YIELD_PATH = r"D:\Crop_yield_system\data\raw\crop_yeild_2011_2022_seasonal.csv"


# ============================================================
# 3. LOAD DATA
# ============================================================
df = pd.read_csv(ENGINEERED_PATH)
df_y = pd.read_csv(RAW_YIELD_PATH)

print("Loaded engineered shape:", df.shape)
print("Loaded yield shape:", df_y.shape)


# ============================================================
# 4. CLEAN COLUMN NAMES
# ============================================================
df.columns = df.columns.str.strip().str.replace(r"\s+", "_", regex=True)
df_y.columns = df_y.columns.str.strip().str.replace(r"\s+", "_", regex=True)

df_y = df_y.rename(columns={
    "Area_(Hectare)": "Area",
    "Production_(Tonnes)": "Production",
    "Yield_(Tonne/Hectare)": "Yield"
})


# ============================================================
# 5. FIX YEAR FORMAT ("2011 - 2012" → 2012)
# ============================================================
def extract_correct_agri_year(value):
    """Extracts the SECOND year from '2011 - 2012' -> 2012."""
    if isinstance(value, str):
        clean = value.replace(" ", "")
        parts = clean.split("-")
        if len(parts) == 2 and parts[1].isdigit():
            return int(parts[1])
    return np.nan

df_y["Year"] = df_y["Year"].apply(extract_correct_agri_year).astype("Int64")
print("Unique years in yield data:", df_y["Year"].unique())


# ============================================================
# 6. CLEAN STATES
# ============================================================
df["State"] = df["State"].astype(str).str.strip()
df_y["State"] = df_y["State"].astype(str).str.strip()

print("\nEngineered dataset states:", df["State"].unique())
print("Yield dataset states:", df_y["State"].unique())


# ============================================================
# 7. REMOVE DUPLICATES BEFORE MERGE
# ============================================================
df_y = df_y.drop_duplicates(subset=["State", "Year"])
print("\nYield dataset after removing duplicates:", df_y.shape)


# ============================================================
# 8. MERGE ENGINEERED FEATURES + YIELD TARGET
# ============================================================
df = df.merge(
    df_y[["State", "Year", "Yield"]],
    on=["State", "Year"],
    how="left"
)

print("\nAfter merging:", df.shape)

df = df.dropna(subset=["Yield"]).reset_index(drop=True)
print("After dropping rows missing Yield:", df.shape)

print(df.head())


# ============================================================
# 9. SET UP FEATURES / TARGET
# ============================================================
y = df["Yield"]
X = df.drop(columns=["Yield"])

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("\nCategorical columns:", cat_cols)
print("Numerical columns:", num_cols)


# ============================================================
# 10. PREPROCESSING PIPELINE
# ============================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


# ============================================================
# 11. STACKING MODEL
# ============================================================
xgb = XGBRegressor(
    n_estimators=600, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)

lgbm = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=400, random_state=42
)

ridge = Ridge(alpha=1.0)

base_models = [
    ("xgb", xgb),
    ("lgbm", lgbm),
    ("rf", rf),
    ("ridge", ridge),
]

meta_model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42
)

stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    passthrough=True,
    cv=5
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("stack", stack_model)
])


# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)


# ============================================================
# 13. EVALUATE MODEL
# ============================================================
y_pred = pipeline.predict(X_test)

rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n==============================")
print("MODEL PERFORMANCE")
print("==============================")
print("Test RMSE:", rmse)
print("Test R²:", r2)


# ============================================================
# 14. CROSS-VALIDATION
# ============================================================
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = -cross_val_score(
    pipeline, X, y, scoring="neg_root_mean_squared_error", cv=cv
)

print("\nCV RMSE mean:", cv_scores.mean())
print("CV RMSE std:", cv_scores.std())


Loaded engineered shape: (2002, 38)
Loaded yield shape: (1827, 5)
df_y unique State+Year rows: 62
Final merged shape: (1836, 39)
Unique years in yield data: <IntegerArray>
[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, <NA>]
Length: 13, dtype: Int64

Engineered dataset states: ['Chandigarh' 'Haryana' 'Punjab' 'Rajasthan' 'Uttar Pradesh']
Yield dataset states: ['Chandigarh' 'Haryana' 'Punjab' 'Rajasthan' 'Uttar Pradesh']

After merging: (1836, 40)


KeyError: ['Yield']

In [33]:
df_y = pd.read_csv(r"D:\Crop_yield_system\data\raw\crop_yeild_2011_2022_seasonal.csv", header=None)
df_y.head(20)


,0,1,2,3,4
0,State,Year,Area (Hectare),Production (Tonnes),Yield (Tonne/Hectare)
1,Chandigarh,2011 - 2012,600,2725,4.54
2,Chandigarh,2012 - 2013,575,2590,4.5
3,Chandigarh,2013 - 2014,575,2600,4.52
4,Chandigarh,2014 - 2015,550,2500,4.55
5,Chandigarh,2015 - 2016,550,2530,4.6
6,Chandigarh,2016 - 2017,550,2585,4.7
7,Chandigarh,2017 - 2018,546,2566,4.7
8,Chandigarh,2018 - 2019,546,2730,5
9,Chandigarh,2019 - 2020,546,2457,4.5


In [9]:
import numpy as np

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

xgb.fit(X_train_encoded, y_train_log)
preds = np.expm1(xgb.predict(X_test_encoded))

rmse = mean_squared_error(y_test, preds)
print("XGBoost RMSE (log target):", rmse)


XGBoost RMSE (log target): 0.42860914906087616


In [16]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    num_leaves=64,
    learning_rate=0.05,
    n_estimators=500,
    min_data_in_leaf=5,
    min_split_gain=0.0,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train_encoded, y_train)
preds = model.predict(X_test_encoded)

from sklearn.metrics import mean_squared_error
rmse_light = mean_squared_error(y_test, preds)
print("LightGBM RMSE:", rmse)


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000449 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2083
[LightGBM] [Info] Number of data points in the train set: 1380, number of used features: 77
[LightGBM] [Info] Start training from score 3.946138
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
LightGBM RMSE: 0.4144802644065124


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train_encoded, y_train)
preds_rf = rf.predict(X_test_encoded)
rmse_rf = mean_squared_error(y_test, preds_rf)

print("RandomForest RMSE:", rmse_rf)


RandomForest RMSE: 0.4155509369252889


In [17]:
print("\nModel Comparison:")
print("RandomForest:", rmse_rf)
print("XGBoost:", rmse_xgb)
print("CatBoost:", rmse_cat)
print("ligthgmb:", rmse_light)



Model Comparison:
RandomForest: 0.4155509369252889
XGBoost: 0.41383914432540553
CatBoost: 0.6645504833995202
ligthgmb: 0.4144802644065124


In [19]:
preds_ensemble = (preds_xgb + preds) / 2
rmse_ensemble = mean_squared_error(y_test, preds_ensemble)
print("Ensemble RMSE:", rmse_ensemble)


Ensemble RMSE: 0.4148042642774567


In [20]:
xgb.fit(X_train_encoded, y_train)
import pandas as pd
import numpy as np

importance = pd.Series(
    xgb.feature_importances_,
    index=X_train_encoded.columns
).sort_values(ascending=True)

print(importance)


SOC_Variability          0.000000
SOC_Range                0.000000
State_Uttar Pradesh      0.000000
Season_Kharif            0.000000
Season_Rabi              0.000000
                           ...   
High_Yield_Flag          0.005661
Year_Since_2000          0.007399
Year_Season_2014_Rabi    0.008097
SOC_Skew                 0.296704
Min_SOC                  0.557264
Length: 78, dtype: float32


In [21]:
low_imp_features = importance[importance < 0.001].index.tolist()
print("Low importance features:", low_imp_features)

X_train_reduced = X_train_encoded.drop(columns=low_imp_features)
X_test_reduced  = X_test_encoded.drop(columns=low_imp_features)


Low importance features: ['SOC_Variability', 'SOC_Range', 'State_Uttar Pradesh', 'Season_Kharif', 'Season_Rabi', 'State_Chandigarh', 'Season_Num', 'NDVI_Level_low', 'State_Punjab', 'State_Rajasthan', 'SOC_Stability', 'Year_Season_2023_Kharif', 'Agro_Zone_Urban', 'Region_North', 'Region_West', 'Agro_Zone_Arid', 'Max_SOC', 'Std_SOC', 'State_Haryana']


In [24]:
xgb.fit(X_train_reduced, y_train)
preds = xgb.predict(X_test_reduced)
rmse = mean_squared_error(y_test, preds)
rmse

0.415222307505668

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

# Train RandomForest
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train_encoded, y_train)

# Train XGBoost
xgb = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.015,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.9,
    min_child_weight=3,
    reg_lambda=1.2,
    reg_alpha=0.1,
    random_state=42
)

xgb.fit(X_train_encoded, y_train)

# Train LightGBM
lgb = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=40,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb.fit(X_train_encoded, y_train)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001762 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2077
[LightGBM] [Info] Number of data points in the train set: 1380, number of used features: 74
[LightGBM] [Info] Start training from score 3.946138
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

,boosting_type,'gbdt'
,num_leaves,40
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [26]:
rf_imp  = pd.Series(rf.feature_importances_,  index=X_train_encoded.columns)
xgb_imp = pd.Series(xgb.feature_importances_, index=X_train_encoded.columns)
lgb_imp = pd.Series(lgb.feature_importances_, index=X_train_encoded.columns)
importance_df = pd.DataFrame({
    "RandomForest": rf_imp,
    "XGBoost": xgb_imp,
    "LightGBM": lgb_imp,
})

# Normalized importance (optional but recommended)
importance_df_norm = importance_df / importance_df.sum()

# Add average rank score
importance_df_norm["AverageScore"] = importance_df_norm.mean(axis=1)

# Sort features by importance
importance_ranked = importance_df_norm.sort_values("AverageScore", ascending=False)

importance_ranked.head(20)   # top 20 features


,RandomForest,XGBoost,LightGBM,AverageScore
Min_SOC,0.445360,0.532210,0.003548,0.327039
SOC_Skew,0.285557,0.305181,0.005039,0.198592
Year,0.046014,0.002673,0.089773,0.046153
High_Yield_Flag,0.081125,0.009175,0.029050,0.039783
Area,0.008821,0.001546,0.084734,0.031700
T2M,0.001858,0.001195,0.087151,0.030068
Production,0.005499,0.001508,0.052496,0.019835
T2M_MAX,0.003927,0.001720,0.051931,0.019193
Year_Since_2000,0.039245,0.007121,0.010386,0.018917
PRECTOTCORR,0.000870,0.002165,0.046738,0.016591


In [28]:
importance_df = pd.DataFrame({
    "RandomForest": rf_imp,
    "XGBoost": xgb_imp,
    "LightGBM": lgb_imp,
})

importance_df_norm = importance_df / importance_df.sum()
importance_df_norm["AverageScore"] = importance_df_norm.mean(axis=1)

importance_ranked = importance_df_norm.sort_values("AverageScore", ascending=False)


In [29]:
low_features = importance_ranked.tail(15)  # bottom 15 features
print(low_features)


                         RandomForest  XGBoost  LightGBM  AverageScore
SOC_Range                2.100926e-04      0.0  0.000051  8.716972e-05
SOC_Stability            1.481612e-04      0.0  0.000000  4.938707e-05
Region_North             1.108553e-04      0.0  0.000000  3.695176e-05
Agro_Zone_Arid           1.072366e-04      0.0  0.000000  3.574552e-05
NDVI_Level_low           1.000354e-04      0.0  0.000000  3.334512e-05
State_Uttar Pradesh      8.624873e-05      0.0  0.000000  2.874958e-05
State_Rajasthan          6.902256e-05      0.0  0.000000  2.300752e-05
Season_Num               5.472658e-05      0.0  0.000000  1.824219e-05
Agro_Zone_Urban          5.370842e-05      0.0  0.000000  1.790281e-05
Season_Kharif            4.580736e-05      0.0  0.000000  1.526912e-05
State_Chandigarh         3.688031e-05      0.0  0.000000  1.229344e-05
State_Punjab             3.299426e-05      0.0  0.000000  1.099809e-05
Season_Rabi              2.915591e-05      0.0  0.000000  9.718635e-06
Region

In [30]:
features_to_drop = low_features.index.tolist()

X_train_reduced = X_train_encoded.drop(columns=features_to_drop)
X_test_reduced  = X_test_encoded.drop(columns=features_to_drop)


In [32]:
xgb.fit(X_train_reduced, y_train)
preds = xgb.predict(X_test_reduced)

from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(y_test, preds)
print("RMSE after dropping features:", rmse)


RMSE after dropping features: 0.41417095839270074
